# 5장 — RoPE, SwiGLU, RMSNorm, KV Cache

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/HisameOgasahara/zerokaraLLM/blob/main/notebooks/ch05_modern_llm_blocks.ipynb)

이 노트북은 『밑바닥부터 시작하는 딥러닝 6』의 공식 코드 저장소를 기준으로 구성했습니다. T4에서 실행하기 어렵다는 이유로 알고리즘이나 모델 구조를 토이 버전으로 바꾸지 않습니다.

- 기준 upstream commit: `c9b6e2ed531b08dd9f451a091a34e9645148e2e2`
- 포함한 장 코드 파일 수: **6개**
- 함께 펼쳐서 보여주는 공통 모듈 수: **0개**


## 노트북 구성 원칙

1. 공식 `.py`의 모델 구조와 계산 로직을 그대로 유지합니다.
2. 함수·클래스·실행부를 셀 단위로 나눠 위에서 아래로 읽기 쉽게 배치합니다.
3. 일본어 자연어 주석은 한국어로 바꾸며, 변수명·수식·텐서 shape 같은 기술 표기는 유지합니다.
4. 공통 `codebot` / `storybot` 모듈도 외부 파일 뒤에 숨기지 않고 이 노트북에서 직접 확인할 수 있게 합니다.
5. T4에서 시간이 오래 걸리는 전체 학습 스케줄도 기본값 자체를 임의 축소하지 않습니다.


## 0. Colab 환경 준비

먼저 공식 저장소를 고정된 커밋으로 준비하고 현재 런타임의 GPU를 확인합니다.


In [ ]:
from pathlib import Path
import os
import subprocess

UPSTREAM_COMMIT = 'c9b6e2ed531b08dd9f451a091a34e9645148e2e2'
WORKDIR = Path('/content/deep-learning-from-scratch-6')

if not WORKDIR.exists():
    subprocess.run(['git', 'clone', '--quiet', 'https://github.com/oreilly-japan/deep-learning-from-scratch-6.git', str(WORKDIR)], check=True)
    subprocess.run(['git', '-C', str(WORKDIR), 'checkout', '--quiet', UPSTREAM_COMMIT], check=True)

os.chdir(WORKDIR)
print('작업 경로:', Path.cwd())

try:
    import torch
    print('PyTorch:', torch.__version__)
    print('CUDA 사용 가능:', torch.cuda.is_available())
    if torch.cuda.is_available():
        print('GPU:', torch.cuda.get_device_name(0))
except Exception as exc:
    print('PyTorch 확인 중 오류:', exc)


## 2. 장별 실습 코드

공식 저장소의 장 코드를 파일 순서대로 모두 다룹니다.


## `ch05/01_rope.py`

원본 스크립트를 노트북 흐름에 맞춰 구성 요소별 셀로 나눴습니다. 실행 코드 자체는 주석을 제외하고 변경하지 않았습니다.


### 필요한 라이브러리와 모듈 불러오기


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F


### `RoPE` 클래스 구현


In [ ]:


class RoPE(nn.Module):
    def __init__(self, theta, key_dim, max_context_len):
        super().__init__()
        assert key_dim % 2 == 0  # 이 코드 단계의 동작을 확인하는 예시
        half = key_dim // 2

        half_ids = torch.arange(0, half)
        inv_freq = 1.0 / (theta ** ( (2.0 * half_ids) / key_dim ))  # 출력 예시: (half,)

        positions = torch.arange(max_context_len)  # 출력 예시: (max_context_len,)
        angles = positions[:, None] * inv_freq[None, :]  # 출력 예시: (max_context_len, half)

        cos = torch.cos(angles)  # 출력 예시: (max_context_len, half)
        sin = torch.sin(angles)  # 출력 예시: (max_context_len, half)

        self.register_buffer("cos_cache", cos)
        self.register_buffer("sin_cache", sin)

    def forward(self, x):
        batch_size, num_head, context_len, key_dim = x.shape

        # 이 코드 단계의 동작을 확인하는 예시
        input_dtype = x.dtype
        x = x.float()

        cos = self.cos_cache[:context_len]
        sin = self.sin_cache[:context_len]

        # 이 코드 단계의 동작을 확인하는 예시
        x_even = x[..., 0::2]
        x_odd  = x[..., 1::2]

        # 이 코드 단계의 동작을 확인하는 예시
        x_rot_even = x_even * cos - x_odd * sin
        x_rot_odd  = x_even * sin + x_odd * cos

        # 이 코드 단계의 동작을 확인하는 예시
        out = torch.stack([x_rot_even, x_rot_odd], dim=-1)  # 출력 예시: (batch_size, num_head, context_len, key_dim/2, 2)
        out = out.reshape(batch_size, num_head, context_len, key_dim)

        return out.to(input_dtype)  # 이 코드 단계의 동작을 확인하는 예시


### `MultiHeadAttention` 클래스 구현


In [ ]:

class MultiHeadAttention(nn.Module):
    def __init__(self, embed_dim, n_head, head_dim, rope=None):
        super().__init__()
        self.n_head = n_head
        self.head_dim = head_dim
        E, H, D = embed_dim, n_head, head_dim

        self.W_q = nn.Linear(E, H*D, bias=False)
        self.W_k = nn.Linear(E, H*D, bias=False)
        self.W_v = nn.Linear(E, H*D, bias=False)
        self.W_o = nn.Linear(H*D, E, bias=False)

        self.rope = rope

    def forward(self, x):
        B, C, E = x.shape
        H, D = self.n_head, self.head_dim

        Q = self.W_q(x)
        K = self.W_k(x)
        V = self.W_v(x)

        Q = Q.view(B, C, H, D).transpose(1, 2)
        K = K.view(B, C, H, D).transpose(1, 2)
        V = V.view(B, C, H, D).transpose(1, 2)

        # RoPE의적용
        if self.rope is not None:
            Q = self.rope(Q)
            K = self.rope(K)

        scores = torch.matmul(Q, K.transpose(-2, -1))
        scores = scores / (D ** 0.5)

        mask = torch.tril(torch.ones(C, C, device=scores.device))
        scores = scores.masked_fill(mask == 0, float('-inf'))

        weights = F.softmax(scores, dim=-1)
        hidden = torch.matmul(weights, V)

        hidden = hidden.transpose(1, 2).contiguous()
        hidden = hidden.view(B, C, H * D)
        output = self.W_o(hidden)
        return output


### 설정 및 값 준비: `embed_dim`


In [ ]:

# 이 코드 단계의 동작을 확인하는 예시
embed_dim = 512


### 설정 및 값 준비: `n_head`


In [ ]:
n_head = 8


### 설정 및 값 준비: `head_dim`


In [ ]:
head_dim = 64


### 설정 및 값 준비: `theta`


In [ ]:
theta = 10000


### 설정 및 값 준비: `max_context_len`


In [ ]:
max_context_len = 1024


### 설정 및 값 준비: `rope`


In [ ]:

# 초기화
rope = RoPE(theta, head_dim, max_context_len)


### 설정 및 값 준비: `mha`


In [ ]:
mha = MultiHeadAttention(embed_dim, n_head, head_dim, rope=rope)


### 설정 및 값 준비: `batch_size`


In [ ]:

# 이 코드 단계의 동작을 확인하는 예시
batch_size = 2


### 설정 및 값 준비: `context_len`


In [ ]:
context_len = 10


### 설정 및 값 준비: `x`


In [ ]:
x = torch.randn(batch_size, context_len, embed_dim)


### 설정 및 값 준비: `output`


In [ ]:

# 이 코드 단계의 동작을 확인하는 예시
output = mha(x)


### 실행 및 결과 확인


In [ ]:
print(output.shape)  # 출력 예시: (2, 10, 512)


## `ch05/02_swiglu.py`

원본 스크립트를 노트북 흐름에 맞춰 구성 요소별 셀로 나눴습니다. 실행 코드 자체는 주석을 제외하고 변경하지 않았습니다.


### 필요한 라이브러리와 모듈 불러오기


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F


### `silu()` 함수 구현


In [ ]:


def silu(x):
    return x * torch.sigmoid(x)


### `SwiGLU` 클래스 구현


In [ ]:

class SwiGLU(nn.Module):
    def __init__(self, x_dim, hidden_dim=None):
        super().__init__()
        if hidden_dim is None:
            hidden_dim = int(x_dim * 8 / 3)

        self.W = nn.Linear(x_dim, hidden_dim, bias=False)
        self.V = nn.Linear(x_dim, hidden_dim, bias=False)
        self.O = nn.Linear(hidden_dim, x_dim, bias=False)

    def forward(self, x):
        a = self.W(x)
        b = self.V(x)

        gated = F.silu(a) * b
        out = self.O(gated)
        return out


## `ch05/03_rmsnorm.py`

원본 스크립트를 노트북 흐름에 맞춰 구성 요소별 셀로 나눴습니다. 실행 코드 자체는 주석을 제외하고 변경하지 않았습니다.


### 필요한 라이브러리와 모듈 불러오기


In [ ]:
import torch
import torch.nn as nn


### `RMSNorm` 클래스 구현


In [ ]:


class RMSNorm(nn.Module):
    def __init__(self, x):
        super().__init__()
        self.gamma = nn.Parameter(torch.ones(x))
        self.eps = 1e-5

    def forward(self, x):
        x2 = x**2
        ms = x2.mean(dim=-1, keepdim=True)
        rms = torch.sqrt(ms + self.eps)
        return self.gamma * x / rms


## `ch05/04_improved_gpt.py`

원본 스크립트를 노트북 흐름에 맞춰 구성 요소별 셀로 나눴습니다. 실행 코드 자체는 주석을 제외하고 변경하지 않았습니다.


### 필요한 라이브러리와 모듈 불러오기


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F


### `RoPE` 클래스 구현


In [ ]:


class RoPE(nn.Module):
    def __init__(self, theta, key_dim, max_context_len):
        super().__init__()
        assert key_dim % 2 == 0
        half = key_dim // 2

        half_ids = torch.arange(0, half)
        inv_freq = 1.0 / (theta ** ( (2.0 * half_ids) / key_dim ))  # 출력 예시: (half,)

        positions = torch.arange(max_context_len)  # 출력 예시: (max_context_len,)
        angles = positions[:, None] * inv_freq[None, :]  # 출력 예시: (max_context_len, half)

        cos = torch.cos(angles)  # 출력 예시: (max_context_len, half)
        sin = torch.sin(angles)  # 출력 예시: (max_context_len, half)

        self.register_buffer("cos_cache", cos)
        self.register_buffer("sin_cache", sin)

    def forward(self, x):
        batch_size, num_head, context_len, key_dim = x.shape

        input_dtype = x.dtype
        x = x.float()

        cos = self.cos_cache[:context_len]
        sin = self.sin_cache[:context_len]

        x_even = x[..., 0::2]
        x_odd  = x[..., 1::2]

        x_rot_even = x_even * cos - x_odd * sin
        x_rot_odd  = x_even * sin + x_odd * cos

        out = torch.stack([x_rot_even, x_rot_odd], dim=-1)
        out = out.reshape(batch_size, num_head, context_len, key_dim)

        return out.to(input_dtype)


### `MultiHeadAttention` 클래스 구현


In [ ]:


class MultiHeadAttention(nn.Module):
    def __init__(self, embed_dim, n_head, head_dim, rope=None):
        super().__init__()
        self.n_head = n_head
        self.head_dim = head_dim
        E, H, D = embed_dim, n_head, head_dim

        self.W_q = nn.Linear(E, H*D, bias=False)
        self.W_k = nn.Linear(E, H*D, bias=False)
        self.W_v = nn.Linear(E, H*D, bias=False)
        self.W_o = nn.Linear(H*D, E, bias=False)

        self.rope = rope

    def forward(self, x):
        B, C, E = x.shape
        H, D = self.n_head, self.head_dim

        Q = self.W_q(x)
        K = self.W_k(x)
        V = self.W_v(x)

        Q = Q.view(B, C, H, D).transpose(1, 2)
        K = K.view(B, C, H, D).transpose(1, 2)
        V = V.view(B, C, H, D).transpose(1, 2)

        # 참고: RoPE
        if self.rope is not None:
            Q = self.rope(Q)
            K = self.rope(K)

        scores = torch.matmul(Q, K.transpose(-2, -1))
        scores = scores / (D ** 0.5)

        mask = torch.tril(torch.ones(C, C, device=scores.device))
        scores = scores.masked_fill(mask == 0, float('-inf'))

        weights = F.softmax(scores, dim=-1)
        hidden = torch.matmul(weights, V)

        hidden = hidden.transpose(1, 2).contiguous()
        hidden = hidden.view(B, C, H * D)
        output = self.W_o(hidden)
        return output


### `silu()` 함수 구현


In [ ]:

def silu(x):
    return x * torch.sigmoid(x)


### `SwiGLU` 클래스 구현


In [ ]:

class SwiGLU(nn.Module):
    def __init__(self, x_dim, hidden_dim=None):
        super().__init__()
        if hidden_dim is None:
            hidden_dim = int(x_dim * 8 / 3)

        self.W = nn.Linear(x_dim, hidden_dim, bias=False)
        self.V = nn.Linear(x_dim, hidden_dim, bias=False)
        self.O = nn.Linear(hidden_dim, x_dim, bias=False)

    def forward(self, x):
        a = self.W(x)
        b = self.V(x)

        gated = F.silu(a) * b  # 참고: silu(a) * b
        out = self.O(gated)
        return out


### `Block` 클래스 구현


In [ ]:

class Block(nn.Module):
    def __init__(self, embed_dim, n_head, ff_dim, rope=None):
        super().__init__()
        head_dim = embed_dim // n_head
        self.norm1 = nn.RMSNorm(embed_dim)
        self.attn = MultiHeadAttention(embed_dim, n_head, head_dim, rope)
        self.norm2 = nn.RMSNorm(embed_dim)
        self.ffn = SwiGLU(embed_dim, ff_dim)

    def forward(self, x):
        x = x + self.attn(self.norm1(x))
        x = x + self.ffn(self.norm2(x))
        return x


### `GPT` 클래스 구현


In [ ]:


class GPT(nn.Module):
    def __init__(self, vocab_size, max_context_len, embed_dim, n_head, n_layer, ff_dim, theta=10000):
        super().__init__()
        self.vocab_size = vocab_size
        self.max_context_len = max_context_len
        self.embed_dim = embed_dim
        self.n_head = n_head
        self.n_layer = n_layer
        self.ff_dim = ff_dim
        self.theta = theta

        self.embed = nn.Embedding(vocab_size, embed_dim)

        head_dim = embed_dim // n_head
        rope = RoPE(theta, head_dim, max_context_len)

        self.blocks = nn.ModuleList([
            Block(embed_dim, n_head, ff_dim, rope)
            for _ in range(n_layer)
        ])

        self.norm = nn.RMSNorm(embed_dim)
        self.unembed = nn.Linear(embed_dim, vocab_size, bias=False)

        self.apply(self._init_weights)

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                torch.nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(self, ids):
        x = self.embed(ids)
        for block in self.blocks:
            x = block(x)
        x = self.norm(x)
        logits = self.unembed(x)  # 출력 예시: (B, C, vocab_size)
        return logits

    def save(self, file_path):
        checkpoint = {
            'model_state_dict': self.state_dict(),
            'vocab_size': self.vocab_size,
            'max_context_len': self.max_context_len,
            'embed_dim': self.embed_dim,
            'n_head': self.n_head,
            'n_layer': self.n_layer,
            'ff_dim': self.ff_dim,
            'theta': self.theta,
        }
        torch.save(checkpoint, file_path)

    @classmethod
    def load_from(cls, file_path, device='cpu'):
        checkpoint = torch.load(file_path, map_location=device)

        model = cls(
            vocab_size=checkpoint['vocab_size'],
            max_context_len=checkpoint['max_context_len'],
            embed_dim=checkpoint['embed_dim'],
            n_head=checkpoint['n_head'],
            n_layer=checkpoint['n_layer'],
            ff_dim=checkpoint['ff_dim'],
            theta=checkpoint['theta']
        )

        model.load_state_dict(checkpoint['model_state_dict'])
        model.to(device)

        return model


### 설정 및 값 준비: `vocab_size`


In [ ]:


# 이 코드 단계의 동작을 확인하는 예시
vocab_size = 10000


### 설정 및 값 준비: `max_context_len`


In [ ]:
max_context_len = 256


### 설정 및 값 준비: `embed_dim`


In [ ]:
embed_dim = 384


### 설정 및 값 준비: `n_head`


In [ ]:
n_head = 6


### 설정 및 값 준비: `n_layer`


In [ ]:
n_layer = 6


### 설정 및 값 준비: `ff_dim`


In [ ]:
ff_dim = int(embed_dim * 8 / 3)


### 설정 및 값 준비: `theta`


In [ ]:
theta = 10000


### 설정 및 값 준비: `model`


In [ ]:

# 모델의초기화
model = GPT(vocab_size, max_context_len, embed_dim, n_head,
            n_layer, ff_dim, theta)


### 설정 및 값 준비: `batch_size`


In [ ]:
# 이 코드 단계의 동작을 확인하는 예시
batch_size = 8


### 설정 및 값 준비: `dummy_input`


In [ ]:
dummy_input = torch.randint(0, vocab_size, (batch_size, max_context_len))


### 설정 및 값 준비: `logits`


In [ ]:
logits = model(dummy_input)


### 실행 및 결과 확인


In [ ]:
print(logits.shape)  # 참고: torch.Size([8, 256, 10000])


## `ch05/05_kvcache.py`

원본 스크립트를 노트북 흐름에 맞춰 구성 요소별 셀로 나눴습니다. 실행 코드 자체는 주석을 제외하고 변경하지 않았습니다.


### 필요한 라이브러리와 모듈 불러오기


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import time


### `RoPE` 클래스 구현


In [ ]:


class RoPE(nn.Module):
    def __init__(self, theta, key_dim, max_context_len):
        super().__init__()
        assert key_dim % 2 == 0
        half = key_dim // 2

        half_ids = torch.arange(0, half)
        inv_freq = 1.0 / (theta ** ( (2.0 * half_ids) / key_dim ))  # 출력 예시: (half,)

        positions = torch.arange(max_context_len)  # 출력 예시: (max_context_len,)
        angles = positions[:, None] * inv_freq[None, :]  # 출력 예시: (max_context_len, half)

        cos = torch.cos(angles)  # 출력 예시: (max_context_len, half)
        sin = torch.sin(angles)  # 출력 예시: (max_context_len, half)

        self.register_buffer("cos_cache", cos)
        self.register_buffer("sin_cache", sin)

    def forward(self, x, offset=0):
        batch_size, num_head, context_len, key_dim = x.shape

        input_dtype = x.dtype
        x = x.float()

        # 이 코드 단계의 동작을 확인하는 예시
        max_context_len = self.cos_cache.size(0)
        if offset + context_len > max_context_len:
            offset = max_context_len - context_len

        cos = self.cos_cache[offset:offset + context_len]
        sin = self.sin_cache[offset:offset + context_len]

        x_even = x[..., 0::2]
        x_odd  = x[..., 1::2]

        x_rot_even = x_even * cos - x_odd * sin
        x_rot_odd  = x_even * sin + x_odd * cos

        out = torch.stack([x_rot_even, x_rot_odd], dim=-1)
        out = out.reshape(batch_size, num_head, context_len, key_dim)

        return out.to(input_dtype)


### `MultiHeadAttention` 클래스 구현


In [ ]:


class MultiHeadAttention(nn.Module):
    def __init__(self, embed_dim, n_head, head_dim, rope=None):
        super().__init__()
        self.n_head = n_head
        self.head_dim = head_dim
        E, H, D = embed_dim, n_head, head_dim

        self.W_q = nn.Linear(E, H*D, bias=False)
        self.W_k = nn.Linear(E, H*D, bias=False)
        self.W_v = nn.Linear(E, H*D, bias=False)
        self.W_o = nn.Linear(H*D, E, bias=False)

        self.rope = rope

        # 이 코드 단계의 동작을 확인하는 예시
        self.k_cache = None  # Key의캐시
        self.v_cache = None  # Value의캐시
        self.cache_offset = 0  # 이 코드 단계의 동작을 확인하는 예시

    def forward(self, x, use_cache=False):
        B, C, E = x.shape
        H, D = self.n_head, self.head_dim

        Q = self.W_q(x)
        K = self.W_k(x)
        V = self.W_v(x)

        Q = Q.view(B, C, H, D).transpose(1, 2)
        K = K.view(B, C, H, D).transpose(1, 2)
        V = V.view(B, C, H, D).transpose(1, 2)

        # 이 코드 단계의 동작을 확인하는 예시
        if self.rope is not None:
            if use_cache:
                Q = self.rope(Q, self.cache_offset)
                K = self.rope(K, self.cache_offset)
            else:
                Q = self.rope(Q)
                K = self.rope(K)

        # KV-Cache의처리
        if use_cache:
            # 이 코드 단계의 동작을 확인하는 예시
            is_first_call = (self.k_cache is None)

            if is_first_call:
                # 이 코드 단계의 동작을 확인하는 예시
                self.k_cache = K
                self.v_cache = V
            else:
                # 이 코드 단계의 동작을 확인하는 예시
                self.k_cache = torch.cat([self.k_cache, K], dim=2)
                self.v_cache = torch.cat([self.v_cache, V], dim=2)

            # 이 코드 단계의 동작을 확인하는 예시
            self.cache_offset += C

            # 이 코드 단계의 동작을 확인하는 예시
            K = self.k_cache
            V = self.v_cache

        # 이 코드 단계의 동작을 확인하는 예시
        scores = torch.matmul(Q, K.transpose(-2, -1))
        scores = scores / (D ** 0.5)

        # 이 코드 단계의 동작을 확인하는 예시
        if not use_cache or is_first_call:
            mask = torch.tril(torch.ones(C, C, device=scores.device))
            scores = scores.masked_fill(mask == 0, float('-inf'))

        weights = F.softmax(scores, dim=-1)
        hidden = torch.matmul(weights, V)

        hidden = hidden.transpose(1, 2).contiguous()
        hidden = hidden.view(B, C, H * D)
        output = self.W_o(hidden)
        return output

    def clear_cache(self):
        """キャッシュをクリアする"""
        self.k_cache = None
        self.v_cache = None
        self.cache_offset = 0


### `silu()` 함수 구현


In [ ]:

def silu(x):
    return x * torch.sigmoid(x)


### `SwiGLU` 클래스 구현


In [ ]:

class SwiGLU(nn.Module):
    def __init__(self, x_dim, hidden_dim=None):
        super().__init__()
        if hidden_dim is None:
            hidden_dim = int(x_dim * 8 / 3)

        self.W = nn.Linear(x_dim, hidden_dim, bias=False)
        self.V = nn.Linear(x_dim, hidden_dim, bias=False)
        self.O = nn.Linear(hidden_dim, x_dim, bias=False)

    def forward(self, x):
        a = self.W(x)
        b = self.V(x)

        gated = F.silu(a) * b  # 참고: silu(a) * b
        out = self.O(gated)
        return out


### `Block` 클래스 구현


In [ ]:

class Block(nn.Module):
    def __init__(self, embed_dim, n_head, ff_dim, rope=None):
        super().__init__()
        head_dim = embed_dim // n_head
        self.norm1 = nn.RMSNorm(embed_dim)
        self.attn = MultiHeadAttention(embed_dim, n_head, head_dim, rope)
        self.norm2 = nn.RMSNorm(embed_dim)
        self.ffn = SwiGLU(embed_dim, ff_dim)

    def forward(self, x, use_cache=False):
        x = x + self.attn(self.norm1(x), use_cache=use_cache)
        x = x + self.ffn(self.norm2(x))
        return x

    def clear_cache(self):
        """キャッシュをクリアする"""
        self.attn.clear_cache()


### `GPT` 클래스 구현


In [ ]:


class GPT(nn.Module):
    def __init__(self, vocab_size, max_context_len, embed_dim, n_head, n_layer, ff_dim, theta=10000):
        super().__init__()
        self.vocab_size = vocab_size
        self.max_context_len = max_context_len
        self.embed_dim = embed_dim
        self.n_head = n_head
        self.n_layer = n_layer
        self.ff_dim = ff_dim
        self.theta = theta

        self.embed = nn.Embedding(vocab_size, embed_dim)

        head_dim = embed_dim // n_head
        rope = RoPE(theta, head_dim, max_context_len)

        self.blocks = nn.ModuleList([
            Block(embed_dim, n_head, ff_dim, rope)
            for _ in range(n_layer)
        ])

        self.norm = nn.RMSNorm(embed_dim)
        self.unembed = nn.Linear(embed_dim, vocab_size, bias=False)

        self.apply(self._init_weights)

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                torch.nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(self, ids, use_cache=False):
        x = self.embed(ids)
        for block in self.blocks:
            x = block(x, use_cache=use_cache)
        x = self.norm(x)
        logits = self.unembed(x)
        return logits

    def save(self, file_path):
        checkpoint = {
            'model_state_dict': self.state_dict(),
            'vocab_size': self.vocab_size,
            'max_context_len': self.max_context_len,
            'embed_dim': self.embed_dim,
            'n_head': self.n_head,
            'n_layer': self.n_layer,
            'ff_dim': self.ff_dim,
            'theta': self.theta,
        }
        torch.save(checkpoint, file_path)

    @classmethod
    def load_from(cls, file_path, device='cpu'):
        checkpoint = torch.load(file_path, map_location=device)

        model = cls(
            vocab_size=checkpoint['vocab_size'],
            max_context_len=checkpoint['max_context_len'],
            embed_dim=checkpoint['embed_dim'],
            n_head=checkpoint['n_head'],
            n_layer=checkpoint['n_layer'],
            ff_dim=checkpoint['ff_dim'],
            theta=checkpoint['theta']
        )

        model.load_state_dict(checkpoint['model_state_dict'])
        model.to(device)

        return model

    def clear_cache(self):
        """全てのブロックのキャッシュをクリアする"""
        for block in self.blocks:
            block.clear_cache()


### `generate_without_cache()` 함수 구현


In [ ]:


def generate_without_cache(model, start_ids, max_new_tokens):
    model.eval()

    ids = start_ids  # 이 코드 단계의 동작을 확인하는 예시
    with torch.no_grad():
        for _ in range(max_new_tokens):
            # 이 코드 단계의 동작을 확인하는 예시
            logits = model(ids, use_cache=False)
            next_id = torch.argmax(logits[:, -1, :], dim=-1, keepdim=True)
            # 이 코드 단계의 동작을 확인하는 예시
            ids = torch.cat([ids, next_id], dim=1)

    return ids


### `generate_with_cache()` 함수 구현


In [ ]:

def generate_with_cache(model, start_ids, max_new_tokens):
    model.eval()

    ids = start_ids  # 이 코드 단계의 동작을 확인하는 예시
    next_id = start_ids
    with torch.no_grad():
        for _ in range(max_new_tokens):
            # 이 코드 단계의 동작을 확인하는 예시
            logits = model(next_id, use_cache=True)
            next_id = torch.argmax(logits[:, -1, :], dim=-1, keepdim=True)
            ids = torch.cat([ids, next_id], dim=1)
    return ids


### `measure_generation_time()` 함수 구현


In [ ]:

def measure_generation_time(model, start_ids, use_cache, num_tokens=200):
    if not use_cache:
        model.clear_cache()

    start_time = time.time()

    if use_cache:
        generate_with_cache(model, start_ids, num_tokens)
    else:
        generate_without_cache(model, start_ids, num_tokens)

    elapsed = time.time() - start_time
    return elapsed


### 설정 및 값 준비: `model`


In [ ]:

# 이 코드 단계의 동작을 확인하는 예시
model = GPT(vocab_size=1000, max_context_len=256, embed_dim=384,
            n_head=6, n_layer=6, ff_dim=1024)


### 설정 및 값 준비: `start_ids`


In [ ]:


start_ids = torch.tensor([[42]])  # 이 코드 단계의 동작을 확인하는 예시


### 설정 및 값 준비: `time_without`


In [ ]:

time_without = measure_generation_time(model, start_ids, use_cache=False)


### 설정 및 값 준비: `time_with`


In [ ]:
time_with = measure_generation_time(model, start_ids, use_cache=True)


### 실행 및 결과 확인


In [ ]:
print(f"KV-Cacheなし: {time_without:.2f}秒")


### 실행 및 결과 확인


In [ ]:
print(f"KV-Cacheあり: {time_with:.2f}秒")


### 실행 및 결과 확인


In [ ]:
print(f"高速化率: {time_without / time_with:.1f}倍")


### 실행 및 결과 확인


In [ ]:

"""
print("\n=== 出力の一致確認 ===")
model.clear_cache()

# 同じ開始トークンで生成

# KV-Cacheなしで生成
output_without = generate_without_cache(model, start_ids, max_new_tokens=max_new_tokens)
print(f"KV-Cacheなし: {output_without[0, :11].tolist()}")

# KV-Cacheありで生成
model.clear_cache()
output_with = generate_with_cache(model, start_ids, max_new_tokens=max_new_tokens)

print(f"KV-Cacheあり: {output_with[0, :11].tolist()}")

print(output_with.shape, output_without.shape)
# 一致確認
if torch.equal(output_without[:, :max_new_tokens], output_with[:, :max_new_tokens]):
    print("✓ 出力が一致しました!")
else:
    print("✗ 出力が一致しません")
    print(f"差分の数: {(output_without[:, :max_new_tokens] != output_with[:, :max_new_tokens]).sum().item()}")
"""


## `ch05/graph.py`

원본 스크립트를 노트북 흐름에 맞춰 구성 요소별 셀로 나눴습니다. 실행 코드 자체는 주석을 제외하고 변경하지 않았습니다.


### 필요한 라이브러리와 모듈 불러오기


In [ ]:
import matplotlib.pyplot as plt
import numpy as np


### 설정 및 값 준비: `x`


In [ ]:

# 데이터 생성
x = np.linspace(-3, 3, 500)


### 설정 및 값 준비: `relu`


In [ ]:

# ReLU함수
relu = np.maximum(0, x)


### 설정 및 값 준비: `gelu`


In [ ]:

# GELU함수（근사식）
gelu = 0.5 * x * (1 + np.tanh(np.sqrt(2 / np.pi) * (x + 0.044715 * x**3)))


### 설정 및 값 준비: `swish`


In [ ]:

# Swish함수（SiLU）
swish = x * (1 / (1 + np.exp(-x)))  # 참고: x * sigmoid(x)


### 실행 코드


In [ ]:

# 이 코드 단계의 동작을 확인하는 예시
fig, ax = plt.subplots(figsize=(10, 6))


### 실행 및 결과 확인


In [ ]:

ax.plot(x, relu, 'b-', linewidth=2, label='ReLU')


### 실행 및 결과 확인


In [ ]:
ax.plot(x, gelu, 'g-', linewidth=2, label='GELU')


### 실행 및 결과 확인


In [ ]:
ax.plot(x, swish, 'r-', linewidth=2, label='Swish')


### 실행 및 결과 확인


In [ ]:

# 축 설정
ax.set_xlim(-3, 3)


### 실행 및 결과 확인


In [ ]:
ax.set_ylim(-0.5, 3.0)


### 실행 및 결과 확인


In [ ]:
ax.set_xlabel('x', fontsize=12)


### 실행 및 결과 확인


In [ ]:
ax.set_ylabel('f(x)', fontsize=12)


### 실행 및 결과 확인


In [ ]:

# 격자
ax.grid(True, linestyle='--', alpha=0.7)


### 실행 및 결과 확인


In [ ]:
ax.axhline(y=0, color='gray', linewidth=0.5)


### 실행 및 결과 확인


In [ ]:
ax.axvline(x=0, color='gray', linewidth=0.5)


### 실행 및 결과 확인


In [ ]:

# 범례
ax.legend(loc='upper left', fontsize=12)


### 실행 및 결과 확인


In [ ]:

# 여백 조정
plt.tight_layout()


### 실행 및 결과 확인


In [ ]:

# 이 코드 단계의 동작을 확인하는 예시
plt.savefig('activation_comparison.png', format='png', bbox_inches='tight')


### 실행 및 결과 확인


In [ ]:
plt.close()


## T4 실행 메모

위 코드는 공식 구현의 모델 구조·알고리즘·기본 하이퍼파라미터를 보존합니다. 학습 시간이 긴 셀은 T4에서도 실행 자체는 가능할 수 있지만 전체 스텝 완주에는 시간이 많이 필요할 수 있습니다. 이 노트북은 빠른 실행을 위해 모델을 임의로 축소하거나 핵심 계산을 생략하지 않습니다.
